In [17]:
import pandas as pd

df = pd.read_csv(
    "/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

print(df.shape)
display(df.head())

(180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [18]:

# Step 1: Check dataset information
print("DATASET INFORMATION")
df.info()

# Step 2: Check missing values
print("\nMISSING VALUES")
print(
    df.isnull().sum()
    .sort_values(ascending=False)
    .head(15)
)

# Step 3: Check duplicate rows
print("\nDUPLICATE ROWS")
print(df.duplicated().sum())

# Step 4: Check dataset size
print("\nTOTAL ROWS:", len(df))
print("TOTAL COLUMNS:", len(df.columns))

# Step 5: Check important columns
print("\nIMPORTANT COLUMNS")
for col in df.columns:
    if any(word in col.lower()
           for word in ["order", "sales", "shipping",
                        "delivery", "date"]):
        print(col)


DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 53 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Email                 180519 non-null  object 
 12  Customer F

In [19]:

# Create a working copy
clean_df = df.copy()

print("Original shape:", df.shape)

# Remove personal and unnecessary columns
columns_to_remove = [
    "Customer Email",
    "Customer Fname",
    "Customer Lname",
    "Customer Password",
    "Customer Street",
    "Customer Zipcode",
    "Latitude",
    "Longitude",
    "Product Description",
    "Order Zipcode",
    "Product Image"
]

clean_df = clean_df.drop(
    columns=columns_to_remove,
    errors="ignore"
)

print("After removing columns:", clean_df.shape)


Original shape: (180519, 53)
After removing columns: (180519, 42)


In [20]:

# Convert order and shipping dates
clean_df["order_date"] = pd.to_datetime(
    clean_df["order date (DateOrders)"],
    errors="coerce"
)

clean_df["shipping_date"] = pd.to_datetime(
    clean_df["shipping date (DateOrders)"],
    errors="coerce"
)

print("Invalid order dates:",
      clean_df["order_date"].isna().sum())

print("Invalid shipping dates:",
      clean_df["shipping_date"].isna().sum())

display(
    clean_df[["order_date", "shipping_date"]].head()
)


Invalid order dates: 0
Invalid shipping dates: 0


,order_date,shipping_date
0,2018-01-31 22:56:00,2018-02-03 22:56:00
1,2018-01-13 12:27:00,2018-01-18 12:27:00
2,2018-01-13 12:06:00,2018-01-17 12:06:00
3,2018-01-13 11:45:00,2018-01-16 11:45:00
4,2018-01-13 11:24:00,2018-01-15 11:24:00


In [21]:

# STEP 4: FIVE DATA QUALITY CHECKS

print("SUPPLY CHAIN DATA QUALITY REPORT")
print("-" * 45)

# Check 1: Missing Order Item IDs
missing_ids = clean_df["Order Item Id"].isna().sum()

# Check 2: Duplicate Order Item IDs
duplicate_ids = clean_df["Order Item Id"].duplicated().sum()

# Check 3: Invalid Sales Values
invalid_sales = (
    clean_df["Sales"].isna() |
    (clean_df["Sales"] < 0)
).sum()

# Check 4: Invalid Delivery Dates
invalid_dates = (
    clean_df["order_date"].isna() |
    clean_df["shipping_date"].isna() |
    (clean_df["shipping_date"] <
     clean_df["order_date"])
).sum()

# Check 5: Missing or Blank Shipping Modes
shipping_mode = clean_df["Shipping Mode"].astype("string")
missing_modes = (
    shipping_mode.isna() |
    shipping_mode.str.strip().eq("")
).sum()

# Display results
print("1. Missing Order Item IDs:", missing_ids)
print("2. Duplicate Order Item IDs:", duplicate_ids)
print("3. Invalid Sales Values:", invalid_sales)
print("4. Invalid Delivery Dates:", invalid_dates)
print("5. Missing Shipping Modes:", missing_modes)

print("-" * 45)
print("Total Records:", len(clean_df))


SUPPLY CHAIN DATA QUALITY REPORT
---------------------------------------------
1. Missing Order Item IDs: 0
2. Duplicate Order Item IDs: 0
3. Invalid Sales Values: 0
4. Invalid Delivery Dates: 0
5. Missing Shipping Modes: 0
---------------------------------------------
Total Records: 180519


In [22]:

# STEP 5: CREATE CLEANED DATASET

# Keep original working data unchanged
delivery_df = clean_df.copy()

# Identify records with valid shipping information
valid_records = (
    delivery_df["order_date"].notna() &
    delivery_df["shipping_date"].notna() &
    (delivery_df["shipping_date"] >=
     delivery_df["order_date"]) &
    delivery_df["Shipping Mode"].notna() &
    delivery_df["Shipping Mode"].astype("string")
        .str.strip().ne("").fillna(False)
)

# Keep valid records
delivery_df = delivery_df.loc[valid_records].copy()

# Display results
print("Original records:", len(clean_df))
print("Cleaned records:", len(delivery_df))
print("Removed records:",
      len(clean_df) - len(delivery_df))

# Save cleaned dataset
delivery_df.to_csv(
    "supply_chain_cleaned.csv",
    index=False
)

print("Cleaned CSV saved successfully!")


Original records: 180519
Cleaned records: 180519
Removed records: 0
Cleaned CSV saved successfully!


In [23]:

import sqlite3
import pandas as pd

# Create SQLite database
conn = sqlite3.connect("supply_chain.db")

# Load cleaned dataset into SQLite
delivery_df.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

# Check SQL records
result = pd.read_sql_query(
    "SELECT COUNT(*) AS total_records FROM orders",
    conn
)

display(result)


,total_records
0,180519


In [24]:

# SQL Query: Late Delivery Sales Analysis

query = """
SELECT
    COUNT(*) AS total_order_lines,

    ROUND(SUM(Sales), 2) AS total_sales,

    ROUND(SUM(
        CASE
            WHEN "Delivery Status" = 'Late delivery'
            THEN Sales
            ELSE 0
        END
    ), 2) AS late_delivery_sales,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Delivery Status" = 'Late delivery'
                THEN Sales
                ELSE 0
            END
        ) / NULLIF(SUM(Sales), 0),
        2
    ) AS late_sales_percentage

FROM orders
"""

result = pd.read_sql_query(query, conn)

display(result)


,total_order_lines,total_sales,late_delivery_sales,late_sales_percentage
0,180519,36784735.01,20126395.27,54.71


In [25]:

# SQL: Shipping Mode Performance Analysis

query = """
SELECT
    "Shipping Mode" AS shipping_mode,

    COUNT(*) AS total_order_lines,

    SUM(
        CASE
            WHEN "Delivery Status" = 'Late delivery'
            THEN 1 ELSE 0
        END
    ) AS late_order_lines,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Delivery Status" = 'Late delivery'
                THEN 1 ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS late_delivery_percentage,

    ROUND(
        100.0 * SUM(
            CASE
                WHEN "Delivery Status" IN
                ('Shipping on time', 'Advance shipping')
                THEN 1 ELSE 0
            END
        ) / COUNT(*),
        2
    ) AS on_time_percentage

FROM orders

GROUP BY "Shipping Mode"

ORDER BY on_time_percentage DESC
"""

shipping_results = pd.read_sql_query(query, conn)

display(shipping_results)


,shipping_mode,total_order_lines,late_order_lines,late_delivery_percentage,on_time_percentage
0,Standard Class,107752,41023,38.07,57.66
1,Same Day,9737,4454,45.74,49.70
2,Second Class,35216,26987,76.63,19.36
3,First Class,27814,26513,95.32,0.00


In [26]:

# STEP 6.4: Validate delivery status categories

query = """
SELECT
    "Shipping Mode" AS shipping_mode,
    "Delivery Status" AS delivery_status,
    COUNT(*) AS order_lines
FROM orders
GROUP BY
    "Shipping Mode",
    "Delivery Status"
ORDER BY
    "Shipping Mode",
    order_lines DESC
"""

status_results = pd.read_sql_query(query, conn)

display(status_results)


,shipping_mode,delivery_status,order_lines
0,First Class,Late delivery,26513
1,First Class,Shipping canceled,1301
2,Same Day,Shipping on time,4839
3,Same Day,Late delivery,4454
4,Same Day,Shipping canceled,444
5,Second Class,Late delivery,26987
6,Second Class,Shipping on time,6819
7,Second Class,Shipping canceled,1410
8,Standard Class,Advance shipping,41592
9,Standard Class,Late delivery,41023


In [27]:

# STEP 6.5: Final On-Time Delivery Analysis

query = """
SELECT
    "Shipping Mode" AS shipping_mode,

    COUNT(*) AS total_order_lines,

    SUM(CASE
        WHEN "Delivery Status" = 'Shipping canceled'
        THEN 1 ELSE 0
    END) AS canceled_lines,

    SUM(CASE
        WHEN "Delivery Status" IN
            ('Shipping on time', 'Advance shipping')
        THEN 1 ELSE 0
    END) AS on_time_lines,

    SUM(CASE
        WHEN "Delivery Status" = 'Late delivery'
        THEN 1 ELSE 0
    END) AS late_lines,

    ROUND(
        100.0 * SUM(CASE
            WHEN "Delivery Status" IN
                ('Shipping on time', 'Advance shipping')
            THEN 1 ELSE 0
        END)
        / NULLIF(SUM(CASE
            WHEN "Delivery Status" != 'Shipping canceled'
            THEN 1 ELSE 0
        END), 0),
        2
    ) AS on_time_rate

FROM orders

GROUP BY "Shipping Mode"
ORDER BY on_time_rate DESC
"""

final_shipping = pd.read_sql_query(query, conn)

display(final_shipping)


,shipping_mode,total_order_lines,canceled_lines,on_time_lines,late_lines,on_time_rate
0,Standard Class,107752,4599,62130,41023,60.23
1,Same Day,9737,444,4839,4454,52.07
2,Second Class,35216,1410,6819,26987,20.17
3,First Class,27814,1301,0,26513,0.00


In [28]:

# STEP 7.1: Regional Monthly Analysis

analysis_df = delivery_df[
    delivery_df["Delivery Status"].isin([
        "Late delivery",
        "Shipping on time",
        "Advance shipping"
    ])
].copy()

# Extract month from order date
analysis_df["order_month"] = (
    analysis_df["order_date"]
    .dt.to_period("M")
    .astype(str)
)

# Create late-delivery indicator
analysis_df["is_late"] = (
    analysis_df["Delivery Status"]
    == "Late delivery"
).astype(int)

# Group data by region and month
regional_monthly = (
    analysis_df
    .groupby(["Order Region", "order_month"])
    .agg(
        total_order_lines=("Order Item Id", "count"),
        late_order_lines=("is_late", "sum")
    )
    .reset_index()
)

# Calculate late-delivery percentage
regional_monthly["late_delivery_rate"] = (
    regional_monthly["late_order_lines"]
    / regional_monthly["total_order_lines"]
    * 100
)

print("Total region-month combinations:",
      len(regional_monthly))

display(regional_monthly.head(10))


Total region-month combinations: 210


,Order Region,order_month,total_order_lines,late_order_lines,late_delivery_rate
0,Canada,2016-08,39,13,33.333333
1,Canada,2016-09,183,116,63.387978
2,Canada,2016-10,166,63,37.951807
3,Canada,2016-11,211,126,59.715640
4,Canada,2016-12,202,103,50.990099
5,Canada,2017-01,106,47,44.339623
6,Caribbean,2015-01,853,468,54.865182
7,Caribbean,2015-02,898,536,59.688196
8,Caribbean,2015-03,876,531,60.616438
9,Caribbean,2015-04,720,362,50.277778


In [29]:

# STEP 7.2: Z-Score Anomaly Detection

import numpy as np
import pandas as pd

# Calculate z-scores within each region
regional_monthly["regional_mean"] = (
    regional_monthly.groupby("Order Region")[
        "late_delivery_rate"
    ].transform("mean")
)

regional_monthly["regional_std"] = (
    regional_monthly.groupby("Order Region")[
        "late_delivery_rate"
    ].transform("std")
)

regional_monthly["z_score"] = np.where(
    regional_monthly["regional_std"] > 0,
    (
        regional_monthly["late_delivery_rate"]
        - regional_monthly["regional_mean"]
    ) / regional_monthly["regional_std"],
    np.nan
)

# Flag unusually high or low late-delivery rates
regional_monthly["is_anomaly"] = (
    regional_monthly["z_score"].abs() > 2
)

# Extract anomalous region-months
anomalies = regional_monthly[
    regional_monthly["is_anomaly"]
].copy()

print("Total region-months:", len(regional_monthly))
print("Anomalies detected:", len(anomalies))

display(
    anomalies[[
        "Order Region",
        "order_month",
        "total_order_lines",
        "late_delivery_rate",
        "z_score"
    ]].sort_values("z_score", ascending=False)
)


Total region-months: 210
Anomalies detected: 8


,Order Region,order_month,total_order_lines,late_delivery_rate,z_score
158,Southern Europe,2016-08,3,100.000000,2.728689
27,Central America,2015-05,2397,59.324155,2.220112
199,Western Europe,2016-09,24,95.833333,2.183533
85,Northern Europe,2016-10,40,87.500000,2.147426
59,Eastern Asia,2016-11,22,86.363636,2.075695
84,Northern Europe,2016-09,18,27.777778,-2.005490
142,Southeast Asia,2016-04,8,50.000000,-2.606529
198,Western Europe,2016-08,7,0.000000,-2.993266


In [30]:

# STEP 7.3: Validate anomaly sample sizes

# Identify anomalies with at least 30 order lines
reliable_anomalies = anomalies[
    anomalies["total_order_lines"] >= 30
].copy()

print("Original anomalies:", len(anomalies))
print("Anomalies with 30+ order lines:",
      len(reliable_anomalies))

display(
    reliable_anomalies[[
        "Order Region",
        "order_month",
        "total_order_lines",
        "late_delivery_rate",
        "z_score"
    ]]
)


Original anomalies: 8
Anomalies with 30+ order lines: 2


,Order Region,order_month,total_order_lines,late_delivery_rate,z_score
27,Central America,2015-05,2397,59.324155,2.220112
85,Northern Europe,2016-10,40,87.500000,2.147426


In [32]:

from google.colab import files

# Export cleaned data
delivery_df.to_csv(
    "supply_chain_cleaned.csv",
    index=False
)

# Export regional monthly analysis
regional_monthly.to_csv(
    "regional_monthly_analysis.csv",
    index=False
)

# Download both files
files.download("supply_chain_cleaned.csv")
files.download("regional_monthly_analysis.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:

import pandas as pd
import os

file_name = "supply_chain_cleaned.csv"

if os.path.exists(file_name):
    check_df = pd.read_csv(file_name, encoding="latin1")

    print("FILE NAME:", file_name)
    print("TOTAL ROWS:", len(check_df))
    print("TOTAL COLUMNS:", len(check_df.columns))
    print("TOTAL SALES:", round(check_df["Sales"].sum(), 2))
else:
    print("File not found in Colab.")


FILE NAME: supply_chain_cleaned.csv
TOTAL ROWS: 180519
TOTAL COLUMNS: 44
TOTAL SALES: 36784735.01


In [34]:

print("Original dataset rows:", len(df))
print("Cleaned CSV rows:", len(check_df))

print("\nOriginal dataset sales:", round(df["Sales"].sum(), 2))
print("Cleaned CSV sales:", round(check_df["Sales"].sum(), 2))

print("\nOriginal duplicate Order Item IDs:",
      df["Order Item Id"].duplicated().sum())

print("Cleaned CSV duplicate Order Item IDs:",
      check_df["Order Item Id"].duplicated().sum())


Original dataset rows: 180519
Cleaned CSV rows: 180519

Original dataset sales: 36784735.01
Cleaned CSV sales: 36784735.01

Original duplicate Order Item IDs: 0
Cleaned CSV duplicate Order Item IDs: 0


In [35]:

# Calculate delivery performance using the full dataset

data = check_df.copy()

total_rows = len(data)
total_sales = data["Sales"].sum()

late_data = data[
    data["Delivery Status"] == "Late delivery"
]

late_order_lines = len(late_data)
late_sales = late_data["Sales"].sum()

late_sales_percentage = (
    late_sales / total_sales * 100
)

print("SUPPLY CHAIN DELIVERY ANALYSIS")
print("--------------------------------")
print("Total Order Lines:", total_rows)
print("Total Sales: $", round(total_sales, 2))
print("Late Order Lines:", late_order_lines)
print("Late Delivery Sales: $", round(late_sales, 2))
print("Late Sales Percentage:",
      round(late_sales_percentage, 2), "%")


SUPPLY CHAIN DELIVERY ANALYSIS
--------------------------------
Total Order Lines: 180519
Total Sales: $ 36784735.01
Late Order Lines: 98977
Late Delivery Sales: $ 20126395.27
Late Sales Percentage: 54.71 %
